# Data Construction and Cleaning Pipeline

**Capstone Project — Moody's Ratings**  
*Classifying sovereign natural resource dependency across 208 countries*

This notebook constructs the master panel dataset covering 1995–2019, merging data from seven sources:
1. **World Bank Indicators (WBI)** — resource rents, sectoral composition, employment, infrastructure, finance
2. **IMF World Economic Outlook (WEO)** — GDP per capita (PPP), government revenue, public debt, structural balance
3. **IMF Investment and Capital Stock Dataset (ICSD)** — gross fixed capital formation, primary net lending
4. **Economic Complexity Index (ECI)** — dependent variable, from the Atlas of Economic Complexity
5. **Varieties of Democracy (V-Dem)** — governance, rule of law, corruption, political stability
6. **Penn World Table 11.0 (PWT)** — human capital, capital stock, TFP, expenditure shares
7. **CEPII** — landlocked dummy

The output is a long-format and wide-format CSV (`master_data_long.csv`, `master_data_wide.csv`) containing all variables for all countries, before any sample filtering or imputation.

---

## 0. Setup

In [13]:
import os
os.makedirs("intermediary", exist_ok=True)
os.makedirs("Graphics/NB1", exist_ok=True)

import wbgapi as wb       # World Bank API
import pandas as pd
from weo import download, WEO  # IMF WEO data
import requests, io
import rdata               # For reading .RData (V-Dem)
import os
import re

## 1. Variable Selection

All indicator codes and variable names are defined below for each source. This serves as the single point of reference: any variable added or removed from the analysis is changed here. For a full description of each variable's role, see Table 1 in the report.

In [14]:
# ──────────────────────────────────────────────────────────────
# World Bank Indicators
# ──────────────────────────────────────────────────────────────
wb_variables = [
    # Resource rents
    'NY.GDP.TOTL.RT.ZS',     # Total natural resources rents (% of GDP)
    'NY.GDP.MINR.RT.ZS',     # Mineral rents (% of GDP)
    'NY.GDP.NGAS.RT.ZS',     # Natural gas rents (% of GDP)
    'NY.GDP.PETR.RT.ZS',     # Oil rents (% of GDP)

    # Economic structure
    'NV.IND.MANF.ZS',        # Manufacturing, value added (% of GDP)
    'NV.IND.TOTL.ZS',        # Industry (incl. construction), value added (% of GDP)
    'TX.VAL.TECH.MF.ZS',     # High-technology exports (% of manufactured exports)
    'NV.AGR.TOTL.ZS',        # Agriculture, forestry, fishing, value added (% of GDP)
    'NV.SRV.TOTL.ZS',        # Services, value added (% of GDP)

    # Savings and capital
    'NY.ADJ.SVNG.CD',        # Adjusted savings: total (current US$)
    'NY.ADJ.ICTR.GN.ZS',     # Adjusted savings: gross savings (% of GNI)
    'NY.ADJ.DRES.GN.ZS',     # Adjusted savings: natural resources depletion (% of GNI)

    # Finance and debt
    'DT.DOD.DIMF.CD',        # Use of IMF credit (DOD, current US$)
    'FS.AST.PRVT.GD.ZS',     # Domestic credit to private sector (% of GDP)
    'FR.INR.RINR',            # Real interest rate (%)
    'FR.INR.LEND',            # Lending interest rate (%)
    'FP.CPI.TOTL.ZG',        # Inflation, consumer prices (annual %)

    # Employment
    'SL.IND.EMPL.ZS',        # Employment in industry (% of total employment)
    'SL.SRV.EMPL.ZS',        # Employment in services (% of total employment)
    'SL.AGR.EMPL.ZS',        # Employment in agriculture (% of total employment)

    # Infrastructure and demographics
    'EG.ELC.ACCS.ZS',        # Access to electricity (% of population)
    'IT.CEL.SETS.P2',        # Mobile cellular subscriptions (per 100 people)
    'SP.URB.TOTL',            # Urban population (% of total population)
    'SP.DYN.LE00.IN',        # Life expectancy at birth, total (years)
    'SP.DYN.CDRT.IN',        # Death rate, crude (per 1,000 people)

    # Trade
    'NE.TRD.GNFS.ZS',        # Trade (% of GDP)
]

# ──────────────────────────────────────────────────────────────
# IMF World Economic Outlook (subject, unit)
# ──────────────────────────────────────────────────────────────
imf_variables = [
    ("Gross domestic product per capita, constant prices",
     "Purchasing power parity; 2017 international dollar"),
    ("General government revenue",
     "Percent of GDP"),
    ("General government net debt",
     "Percent of GDP"),
    ("General government structural balance",
     "Percent of potential GDP"),
]

# ──────────────────────────────────────────────────────────────
# IMF Investment and Capital Stock Dataset (ICSD)
# Series codes for GFCF components and primary balance
# ──────────────────────────────────────────────────────────────
imf_icsd_variables = [
    'P51G_S13_Q_POGDP_PT.A',     # GFCF, General government, Constant prices, % of GDP
    'P51G_PS_Q_POGDP_PT.A',      # GFCF, Private sector, Constant prices, % of GDP
    'P51G_PUPVT_Q_POGDP_PT.A',   # GFCF, Public-private partnership, Constant prices, % of GDP
    'GPB_S13_POGDP_PT',          # Primary net lending, General government, % of GDP
]

# ──────────────────────────────────────────────────────────────
# Economic Complexity Index (dependent variable)
# ──────────────────────────────────────────────────────────────
eci_variables = ['eci']

# ──────────────────────────────────────────────────────────────
# V-Dem: Varieties of Democracy
# ──────────────────────────────────────────────────────────────
vdem_variables = [
    # High-level democracy indices
    'v2x_polyarchy',       # Electoral democracy index
    'v2x_libdem',          # Liberal democracy index
    'v2x_partipdem',       # Participatory democracy index
    'v2x_delibdem',        # Deliberative democracy index
    'v2x_egaldem',         # Egalitarian democracy index

    # Governance
    'v2xnp_client',        # Clientelism index
    'v2x_corr',            # Political corruption index
    'v2x_rule',            # Rule of law index
    'v2x_accountability',  # Accountability index
    'v2xcl_prpty',         # Property rights
    'e_wbgi_pve',          # Political stability (WGI estimate)
    'e_civil_war',         # Civil war indicator
]

# ──────────────────────────────────────────────────────────────
# Penn World Table 11.0
# Note: cn is in constant local currency; ctfp/cwtfp are indices
# ──────────────────────────────────────────────────────────────
pwt_variables = [
    'hc',     # Human capital index
    'cn',     # Capital stock (national accounts prices)
    'ctfp',   # TFP level (constant national prices, index)
    'cwtfp',  # Welfare-relevant TFP
    'csh_c',  # Share of consumption in GDP
    'csh_i',  # Share of investment in GDP
    'csh_g',  # Share of government spending in GDP
    'delta',  # Capital depreciation rate
]

# ──────────────────────────────────────────────────────────────
# CEPII
# ──────────────────────────────────────────────────────────────
cepii_variables = ['landlocked']

## 2. World Bank Indicators

Data is downloaded via the `wbgapi` Python package, which queries the World Bank API directly. We restrict the query to non-aggregate economies (i.e. real countries, excluding regional and income-group aggregates) and the 1995-2019 window.

In [15]:
import time

def download_wb_indicators(indicators, start_year, end_year, max_retries=3, pause=2):
    """Download WBI indicators with retry logic for unstable API connections."""
    final_rows = []

    # Get all real countries (exclude aggregates)
    economies = [c['id'] for c in wb.economy.list() if not c.get("aggregate", False)]

    for indicator in indicators:
        for attempt in range(1, max_retries + 1):
            try:
                print(f"Downloading {indicator} ... (attempt {attempt})")
                raw = wb.data.fetch(indicator, economy=economies, time=range(start_year, end_year + 1))

                count = 0
                for row in raw:
                    iso = row.get("economy")
                    year = int(row.get("time").replace("YR", ""))
                    value = row.get("value")
                    if iso is None or value is None:
                        continue
                    final_rows.append({
                        "Country Code": iso,
                        "Year": year,
                        "Variable": indicator,
                        "Value": value,
                    })
                    count += 1

                print(f"  -> {count} rows")
                break  # success, move to next indicator

            except Exception as e:
                print(f"  Error: {e}")
                if attempt < max_retries:
                    wait = pause * attempt
                    print(f"  Retrying in {wait}s ...")
                    time.sleep(wait)
                else:
                    print(f"  FAILED after {max_retries} attempts. Skipping {indicator}.")

        time.sleep(1)  # brief pause between indicators to avoid throttling

    return pd.DataFrame(final_rows)


wb_df = download_wb_indicators(wb_variables, start_year=1995, end_year=2019)
print(f"\nWB data: {wb_df.shape[0]:,} rows, {wb_df['Variable'].nunique()} indicators")

WB data: 113,836 rows, 26 indicators


### Country name lookup table

We also build a lookup table mapping ISO3 codes to country names, which is used later to attach readable names to all sources.

In [16]:
# Build ISO3 -> Country Name mapping from the World Bank economy list
all_economies = wb.economy.list()
countries = [c for c in all_economies if not c.get("aggregate", False)]

country_names = pd.DataFrame({
    "Country Code": [c["id"] for c in countries],
    "Country Name": [c["value"] for c in countries],
})

print(f"Country lookup table: {len(country_names)} entries")
country_names.head()

Country lookup table: 217 entries


,Country Code,Country Name
0,ABW,Aruba
1,AFG,Afghanistan
2,AGO,Angola
3,ALB,Albania
4,AND,Andorra


## 3. IMF World Economic Outlook

The `weo` Python package downloads the April 2024 vintage of the World Economic Outlook database. We extract four macroeconomic series: GDP per capita at PPP (constant 2017 international dollars), general government revenue (% of GDP), general government net debt (% of GDP), and the structural fiscal balance (% of potential GDP).

**Note on column naming:** The WEO `.get()` method returns a DataFrame with ISO3 codes as the index and years as columns. After reshaping to long format, the columns are renamed to match our standard schema (`Country Code`, `Year`, `Variable`, `Value`).

In [17]:
# IMF WEO download with retry
for attempt in range(1, 4):
    try:
        print(f"Downloading IMF WEO (attempt {attempt}) ...")
        path, _ = download(2024, "Apr")
        print("  Done.")
        break
    except Exception as e:
        print(f"  Error: {e}")
        if attempt < 3:
            time.sleep(5)
        else:
            raise

w = WEO(path)

frames = []
for subj, unit in imf_variables:
    df = w.get(subj, unit).reset_index().rename(columns={"index": "COUNTRY"})
    df_long = df.melt(id_vars="COUNTRY", var_name="YEAR", value_name="VALUE")
    df_long["INDICATOR"] = subj
    frames.append(df_long)

imf_df = pd.concat(frames, ignore_index=True)

# Clean up: remove the downloaded WEO CSV from the working directory
if os.path.exists("weo_2024_1.csv"):
    os.remove("weo_2024_1.csv")

# Rename to standard schema
imf_df = imf_df.rename(columns={
    'COUNTRY': 'Year',
    'YEAR': 'Country Code',
    'INDICATOR': 'Variable',
    'VALUE': 'Value',
})

print(f"IMF WEO data: {imf_df.shape[0]:,} rows, {imf_df['Variable'].nunique()} indicators")

weo_2024_1.csv 19.2Mb
Downloaded 2024-Apr WEO dataset
IMF WEO data: 39,200 rows, 4 indicators


## 4. IMF Investment and Capital Stock Dataset (ICSD)

Two components are retrieved from the IMF ICSD, both hosted on the project's GitHub repository:

1. **Gross Fixed Capital Formation (GFCF):** Three sector-level series (general government, private sector, public-private partnerships) are summed into a single aggregate GFCF variable per country-year.
2. **Primary net lending:** General government primary balance as a share of GDP, from a separate ICSD file.

In [18]:
# ── GFCF components ──
pattern = "|".join(map(re.escape, imf_icsd_variables))

imf_icsd_df = (
    pd.read_csv(
        "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/main/rawdata/"
        "dataset_2025-12-23T17_49_47.423426845Z_DEFAULT_INTEGRATION_IMF.FAD_ICSD_1.0.0.csv"
    )
    .rename(columns={"COUNTRY": "Country Name", "INDICATOR": "Variable"})
    .loc[lambda df: df["SERIES_CODE"].str.contains(pattern, na=False)]
    .assign(Country_Code=lambda df: df["SERIES_CODE"].str[:3])
    .drop(columns=["DATASET", "SERIES_CODE", "OBS_MEASURE", "FREQUENCY", "SCALE"])
    .rename(columns={"Country_Code": "Country Code"})
)

year_cols = [str(y) for y in range(1995, 2020)]
imf_icsd_df = pd.melt(
    imf_icsd_df,
    id_vars=["Country Code", "Variable"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value",
)

# Sum the three GFCF components into one variable per country-year
imf_icsd_df = (
    imf_icsd_df
    .groupby(["Country Code", "Year"], as_index=False)
    .agg({"Value": "sum"})
)
imf_icsd_df["Variable"] = "Gross fixed capital formation, all, Constant prices, Percent of GDP"

# ── Primary net lending ──
imf_icsd_df_2 = (
    pd.read_csv(
        "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/refs/heads/main/rawdata/"
        "dataset_2026-01-05T21_29_20.743520144Z_DEFAULT_INTEGRATION_IMF.FAD_FM_5.0.0.csv"
    )
    .rename(columns={
        "COUNTRY.ID": "Country Code",
        "INDICATOR.ID": "Variable",
        "TIME_PERIOD": "Year",
        "OBS_VALUE": "Value",
    })
    .drop(columns=["FREQUENCY.ID", "SCALE.ID"])
)
imf_icsd_df_2 = imf_icsd_df_2[
    (imf_icsd_df_2["Year"] >= 1995) & (imf_icsd_df_2["Year"] <= 2019)
]

# Combine GFCF and primary net lending
imf_icsd_df = pd.concat([imf_icsd_df, imf_icsd_df_2], ignore_index=True)
print(f"IMF ICSD data: {imf_icsd_df.shape[0]:,} rows")

IMF ICSD data: 9,163 rows


## 5. Economic Complexity Index (ECI)

The Economic Complexity Index is our dependent variable. It measures how sophisticated and diversified a country's productive structure is by combining information on the diversity of exports a country produces and their ubiquity (i.e. the number of other countries able to produce them). A high ECI indicates that a country exports a wide range of products that few other countries can produce, reflecting deep embedded capabilities (Hausmann et al., 2014).

We source ECI values from the Atlas of Economic Complexity (HS92 classification).

In [19]:
eci_df = (
    pd.read_csv(
        "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/refs/heads/main/"
        "rawdata/growth_proj_eci_rankings.csv"
    )
    .rename(columns={"country_iso3_code": "country_code", "eci_hs92": "eci"})
    .drop(columns=["eci_rank_hs92"])
)

eci_df["Variable"] = "Economic Complexity"
eci_df = eci_df.rename(columns={
    "country_code": "Country Code",
    "year": "Year",
    "eci": "Value",
})
eci_df = eci_df[["Country Code", "Year", "Variable", "Value"]]

print(f"ECI data: {eci_df.shape[0]:,} rows, "
      f"{eci_df['Country Code'].nunique()} countries, "
      f"{eci_df['Year'].min()}-{eci_df['Year'].max()}")

ECI data: 4,179 rows, 145 countries, 1995-2023


## 6. Varieties of Democracy (V-Dem)

The V-Dem dataset provides expert-coded measures of political institutions, governance, and democracy. We use indicators covering rule of law, property rights, political corruption, political stability, civil war, and five high-level democracy indices. The data is loaded from the `vdemdata` R package's `.RData` file hosted on GitHub.

In [ ]:
# Download V-Dem .RData from the vdemdata R package repository
url = "https://raw.githubusercontent.com/vdeminstitute/vdemdata/master/data/vdem.RData"
for attempt in range(1, 4):
    try:
        print(f"Downloading V-Dem .RData (~100MB, attempt {attempt}) ...")
        resp = requests.get(url, timeout=120)
        resp.raise_for_status()
        print("  Done.")
        break
    except Exception as e:
        print(f"  Error: {e}")
        if attempt < 3:
            time.sleep(10)
        else:
            raise

with open("vdem.RData", "wb") as f:
    f.write(resp.content)

vdem_r = rdata.read_rda("vdem.RData")
vdem = vdem_r.get("vdem")
os.remove("vdem.RData")

# Select and rename
var_list = ["country_name", "country_text_id", "year"] + vdem_variables
vdem = vdem[var_list].rename(columns={
    "country_name": "Country Name",
    "country_text_id": "Country Code",
    "year": "Year",
})

vdem_df = pd.melt(
    vdem,
    id_vars=["Country Code", "Year"],
    value_vars=vdem_variables,
    var_name="Variable",
    value_name="Value",
)
vdem_df = vdem_df[vdem_df["Year"] >= 1995]

print(f"V-Dem data: {vdem_df.shape[0]:,} rows, {vdem_df['Variable'].nunique()} indicators")

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/rdata/conversion/_conversion.py:856: UserWarning: Missing constructor for R class "Date". The underlying R object is returned instead.
  warnings.warn(


## 7. Penn World Table 11.0

The Penn World Table provides internationally comparable macroeconomic indicators using purchasing power parity (PPP) adjustments. We extract human capital indices, capital stock, total factor productivity measures, expenditure shares, and the capital depreciation rate.

In [ ]:
url = "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/main/rawdata/pwt110.xlsx"
pwt_df = (
    pd.read_excel(url, engine="openpyxl", sheet_name="Data")
    .rename(columns={"countrycode": "Country Code", "country": "Country Name", "year": "Year"})
)

pwt_df = pwt_df[["Country Code", "Country Name", "Year"] + pwt_variables]

pwt_df = pwt_df.melt(
    id_vars=["Country Code", "Year"],
    value_vars=pwt_variables,
    var_name="Variable",
    value_name="Value",
)
pwt_df = pwt_df[(pwt_df["Year"] >= 1995) & (pwt_df["Year"] <= 2019)]

print(f"PWT data: {pwt_df.shape[0]:,} rows, {pwt_df['Variable'].nunique()} indicators")

PWT data: 37,000 rows, 8 indicators


## 8. CEPII: Landlocked Dummy

A binary indicator for whether a country is landlocked, from the CEPII GeoDist dataset. Landlocked countries face higher trade costs and limited access to maritime transport, which can constrain export diversification. Since this is a time-invariant characteristic, we expand it across all years in the panel.

In [ ]:
url = "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/main/rawdata/geo_cepii.xls"
cepii_df = (
    pd.read_excel(url, sheet_name="geo_cepii")
    .rename(columns={"iso3": "Country Code"})
    .drop_duplicates(subset=["Country Code"])
)

cepii_df = cepii_df[["Country Code"] + cepii_variables]
cepii_df["Variable"] = "Landlocked"
cepii_df = cepii_df.rename(columns={"landlocked": "Value"})

# Expand to all years (time-invariant)
years = pd.DataFrame({"Year": range(1995, 2020)})
cepii_df = cepii_df.merge(years, how="cross")

print(f"CEPII data: {cepii_df.shape[0]:,} rows ({cepii_df['Country Code'].nunique()} countries x 25 years)")

CEPII data: 5,625 rows (225 countries x 25 years)


## 9. Natural Resource Production Data

Production volumes for hydrocarbons (oil, natural gas, coal) and metals (lithium, cobalt, nickel, tin, bauxite, copper, aluminium, zinc, manganese, rare earth, platinum group, vanadium, natural graphite) are loaded from a pre-processed CSV. These are aggregated into two broad resource categories (Hydrocarbons and Metals) and summed by country-year.

Country names are standardised to match World Bank naming conventions to allow merging with the rest of the panel.

In [ ]:
production = (
    pd.read_csv(
        "https://raw.githubusercontent.com/AyaanTigdikar/Capstone/refs/heads/main/"
        "rawdata/production_values_w_prices-EM.csv"
    )
    .drop(columns="Unnamed: 0")
    .rename(columns={"Country": "Country Name"})
)

# Keep only production (not consumption), within 1995-2019
production = production[
    (production["Metric"] != "Consumption")
    & (production["Year"] >= 1995)
    & (production["Year"] <= 2019)
]

# ── Standardise country names to World Bank conventions ──
name_fixes = {
    "Australia ": "Australia",
    "Brazil1": "Brazil",
    "Brunei": "Brunei Darussalam",
    "Canadian Oil Sands: Total": "Canada",
    "Czech Republic": "Czechia",
    "DR Congo": "Congo, Dem. Rep.",
    "Democratic Republic of Congo": "Congo, Dem. Rep.",
    "Egypt": "Egypt, Arab Rep.",
    "India2": "India",
    "Indonesia ": "Indonesia",
    "Iran": "Iran, Islamic Rep.",
    "Mexico ": "Mexico",
    "Republic of Congo ": "Congo, Rep.",
    "Republic of Congo": "Congo, Rep.",
    "Russia": "Russian Federation",
    "South Korea": "Korea, Rep.",
    "Syria": "Syrian Arab Republic",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "Turkey": "Turkiye",
    "US": "United States",
    "Venezuela": "Venezuela, RB",
    "Venezuela: Orinoco Belt": "Venezuela, RB",
    "Vietnam": "Viet Nam",
    "Yemen": "Yemen, Rep.",
}
production["Country Name"] = production["Country Name"].replace(name_fixes)

# ── Classify resources into broad categories ──
hydrocarbons = ["Oil", "Natural Gas", "Coal"]
metals = [
    "Lithium", "Cobalt", "Nickel", "Tin", "Bauxite", "Natural Graphite",
    "Copper", "Aluminium", "Zinc", "Manganese", "Rare Earth",
    "Platinum Group", "Vanadium",
]

def classify_resource(x):
    if x in hydrocarbons:
        return "Hydrocarbures"
    elif x in metals:
        return "Metals"
    else:
        return "Others"

production["Resource Category"] = production["Resource"].apply(classify_resource)

# Merge with country codes and aggregate
production = production.merge(country_names, how="inner", on="Country Name")
production["Variable"] = production["Metric"] + "_" + production["Resource Category"]
production = (
    production
    .groupby(["Country Name", "Country Code", "Year", "Variable"])["Value"]
    .sum()
    .reset_index()
)
production = production.drop(columns="Country Name")

print(f"Production data: {production.shape[0]:,} rows, "
      f"{production['Variable'].nunique()} resource-metric combinations")

Production data: 4,061 rows, 4 resource-metric combinations


## 10. Merging and Final Cleaning

All seven source DataFrames are concatenated into a single long-format panel. Variable codes are then renamed to human-readable labels, and non-country territories (Hong Kong, Macao, Puerto Rico, etc.) are dropped.

In [ ]:
# ── Concatenate all sources ──
final_df = pd.concat(
    [wb_df, eci_df, imf_df, imf_icsd_df, vdem_df, pwt_df, cepii_df, production],
    ignore_index=True,
)

# ── Rename variable codes to descriptive names ──
rename_map = {
    # World Bank
    "NV.IND.MANF.ZS":     "Manufacturing",
    "NV.IND.TOTL.ZS":     "Industry",
    "TX.VAL.TECH.MF.ZS":  "High-tech exports",
    "NV.AGR.TOTL.ZS":     "Agriculture",
    "NV.SRV.TOTL.ZS":     "Services",
    "NY.GDP.TOTL.RT.ZS":  "Total natural resources rents (% of GDP)",
    "NY.GDP.MINR.RT.ZS":  "Mineral rents (% of GDP)",
    "NY.GDP.NGAS.RT.ZS":  "Natural gas rents (% of GDP)",
    "NY.GDP.PETR.RT.ZS":  "Oil rents (% of GDP)",
    "NY.ADJ.SVNG.CD":     "Adjusted savings: total (current US$)",
    "NY.ADJ.ICTR.GN.ZS":  "Adjusted savings: gross savings (% of GNI)",
    "NY.ADJ.DRES.GN.ZS":  "Adjusted savings: natural resources depletion (% of GNI)",
    "DT.DOD.DIMF.CD":     "Use of IMF credit (DOD, current US$)",
    "SL.IND.EMPL.ZS":     "Employment in industry (% of total employment)",
    "SL.SRV.EMPL.ZS":     "Employment in services (% of total employment)",
    "SL.AGR.EMPL.ZS":     "Employment in agriculture (% of total employment)",
    "EG.ELC.ACCS.ZS":     "Access to electricity (% of population)",
    "IT.CEL.SETS.P2":     "Mobile cellular subscriptions (per 100 people)",
    "FS.AST.PRVT.GD.ZS":  "Domestic credit to private sector (% of GDP)",
    "FR.INR.RINR":         "Real interest rate (%)",
    "FR.INR.LEND":         "Lending interest rate (%)",
    "FP.CPI.TOTL.ZG":     "Inflation, consumer prices (annual %)",
    "SP.URB.TOTL":         "Urban population (% of total population)",
    "SP.DYN.LE00.IN":     "Life expectancy at birth, total (years)",
    "NE.TRD.GNFS.ZS":     "Trade (% of GDP)",
    "SP.DYN.CDRT.IN":     "Death rates, crude per 1000 people",

    # ECI
    "Economic Complexity": "Economic Complexity Index",

    # IMF WEO
    "Gross domestic product per capita, constant prices": "GDP per capita (constant prices, PPP)",
    "General government revenue": "Government revenue",

    # IMF ICSD
    "GPB_S13_POGDP_PT": "Primary net lending, General government, Percent of GDP",

    # V-Dem
    "v2x_polyarchy":      "electoral_dem",
    "v2x_libdem":         "liberal_dem",
    "v2x_partipdem":      "participatory_dem",
    "v2x_delibdem":       "deliberative_dem",
    "v2x_egaldem":        "egalitarian_dem",
    "v2xnp_client":       "Clientelism index",
    "v2x_corr":           "Political corruption index",
    "v2x_rule":           "Rule of law index",
    "v2x_accountability": "Accountability index",
    "v2xcl_prpty":        "Property rights",
    "e_wbgi_pve":         "Political stability — estimate",
    "e_civil_war":        "Civil war",

    # PWT
    "hc":    "Human capital index",
    "cn":    "Capital stock (national accounts prices)",
    "ctfp":  "TFP level (constant national prices)",
    "cwtfp": "Welfare-relevant TFP",
    "csh_c": "Share of consumption in GDP",
    "csh_i": "Share of investment in GDP",
    "csh_g": "Share of government spending in GDP",
    "delta": "Capital depreciation rate",

    # CEPII
    "landlocked": "Landlocked",
}

final_df["Variable"] = final_df["Variable"].replace(rename_map)

# ── Attach country names ──
final_df = final_df.merge(country_names, how="left", on="Country Code")

# ── Standardise Year column ──
# Some sources return pandas Period objects; convert everything to int
final_df["Year"] = final_df["Year"].apply(
    lambda x: x.year if hasattr(x, "year") else int(x)
)
final_df = final_df[(final_df["Year"] >= 1995) & (final_df["Year"] <= 2019)]

# ── Remove non-country territories ──
not_countries = [
    "HKG", "MAC", "PRI", "VIR", "GUM", "ASM", "CYM", "BMU",
    "GRL", "MAF", "SXM", "CUW", "ABW", "FRO", "MNP", "PYF",
]
final_df = final_df[~final_df["Country Code"].isin(not_countries)]

print(f"Merged dataset: {final_df.shape[0]:,} rows")
print(f"Countries: {final_df['Country Code'].nunique()}")
print(f"Variables: {final_df['Variable'].nunique()}")
print(f"Years: {final_df['Year'].min()} - {final_df['Year'].max()}")

## 11. Variable Selection for Analysis

Not all downloaded variables are used in the final econometric and ML analysis. The list below flags the subset of variables retained for modelling. This selection is based on theoretical relevance, data coverage, and preliminary correlation analysis (see report Section 3.4).

In [ ]:
# Variables retained for the econometric and ML pipeline
important_vars = [
    # Governance
    "Rule of law index",
    "Political stability — estimate",
    "Property rights",

    # Macroeconomic
    "GDP per capita (constant prices, PPP)",
    "Government revenue",
    "Inflation, consumer prices (annual %)",
    "Lending interest rate (%)",
    "Real interest rate (%)",
    "Share of consumption in GDP",
    "Share of government spending in GDP",
    "Share of investment in GDP",

    # GDP structure
    "Economic Complexity Index",
    "Agriculture",
    "Industry",
    "Manufacturing",
    "Services",

    # Finance
    "Adjusted savings: gross savings (% of GNI)",
    "Gross fixed capital formation, all, Constant prices, Percent of GDP",
    "Capital depreciation rate",

    # Human capital and infrastructure
    "Access to electricity (% of population)",
    "Human capital index",
    "Life expectancy at birth, total (years)",
    "Mobile cellular subscriptions (per 100 people)",
    "Urban population (% of total population)",

    # Resource rents
    "Mineral rents (% of GDP)",
    "Natural gas rents (% of GDP)",
    "Oil rents (% of GDP)",
    "Total natural resources rents (% of GDP)",
]

final_df["Important_vars"] = final_df["Variable"].isin(important_vars).astype(int)
print(f"Variables flagged as important: {len(important_vars)}")
print(f"Total unique variables in dataset: {final_df['Variable'].nunique()}")

## 12. Reshape to Wide Format and Export

The long-format panel is pivoted to wide format (one row per country-year, one column per variable) for use in regression and ML notebooks. Both formats are saved.

In [ ]:
print(f"Unique variables before pivot: {final_df['Variable'].nunique()}")

final_df_wide = final_df.pivot(
    index=["Country Code", "Country Name", "Year"],
    columns="Variable",
    values="Value",
).reset_index()

print(f"Wide dataset shape: {final_df_wide.shape}")
print(f"  = {final_df_wide['Country Code'].nunique()} countries "
      f"x {final_df_wide['Year'].nunique()} years "
      f"+ {final_df_wide.shape[1] - 3} variables")

## 13. Save Outputs

In [ ]:
final_df.to_csv("intermediary/master_data_long.csv", index=False)
final_df_wide.to_csv("intermediary/master_data_wide.csv", index=False)

print("Saved:")
print("  intermediary/master_data_long.csv")
print("  intermediary/master_data_wide.csv")

---

## Summary

| Stage | Description |
|-------|-------------|
| **Sources** | World Bank (26 indicators), IMF WEO (4), IMF ICSD (2 aggregated), ECI (1), V-Dem (12), PWT (8), CEPII (1), Production (resource-metric combos) |
| **Panel** | 1995-2019, all non-aggregate World Bank economies minus 16 non-country territories |
| **Output** | `master_data_long.csv` (long format), `master_data_wide.csv` (wide format, one row per country-year) |
| **Next step** | Missingness diagnostics (`MissingCheck.ipynb`) and imputation (`3_Imputing.ipynb`) |

**Pipeline position:** This is Step 1 of 3 in the data preparation sequence:  
`1_cleaning_master_data.ipynb` → `MissingCheck.ipynb` (diagnostics) → `3_Imputing.ipynb` (sample selection + imputation) → Analysis notebooks

In [ ]:
# ── NB1 SUMMARY ──
print("=" * 70)
print("NB1: DATA CONSTRUCTION SUMMARY")
print("=" * 70)
print(f"\nLong format: {final_df.shape[0]:,} rows")
print(f"Wide format: {final_df_wide.shape}")
print(f"Countries: {final_df['Country Code'].nunique()}")
print(f"Variables: {final_df['Variable'].nunique()}")
print(f"Years: {final_df['Year'].min()} - {final_df['Year'].max()}")
print(f"\nVariables flagged as important: {final_df['Important_vars'].sum():,} rows")
print(f"\nWide format columns:")
for col in sorted(final_df_wide.columns):
    if col not in ['Country Code', 'Country Name', 'Year']:
        miss = final_df_wide[col].isna().sum()
        print(f"  {col}: {miss} missing ({100*miss/len(final_df_wide):.1f}%)")
print(f"\nFiles saved:")
print(f"  intermediary/master_data_long.csv")
print(f"  intermediary/master_data_wide.csv")

In [ ]:
# ── NB1 SUMMARY ──
print("=" * 70)
print("NB1: DATA CONSTRUCTION SUMMARY")
print("=" * 70)
print(f"\nLong format: {final_df.shape[0]:,} rows, {final_df['Variable'].nunique()} variables")
print(f"Wide format: {final_df_wide.shape[0]:,} rows x {final_df_wide.shape[1]} columns")
print(f"Countries: {final_df['Country Code'].nunique()}")
print(f"Years: {final_df['Year'].min()} - {final_df['Year'].max()}")
print(f"\nVariables flagged as important: {final_df[final_df['Important_vars']==1]['Variable'].nunique()}")
print(f"\nSaved:")
print(f"  intermediary/master_data_long.csv")
print(f"  intermediary/master_data_wide.csv")